In [1]:
import pandas as pd
from src import pipelines as P
from src import rnn_model as RM
from pathlib import Path
import re
import torch
import torch.nn as nn
import torchmetrics
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader, random_split
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
V_len = RM.V_len

/home/stachuapa123/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
art = P.line_read('raw_data/de-en.txt') # python dictionary
df_art = P.build_dataframe(art) #pandas df
X, y = P.makeXy(df_art)

In [3]:
datset = RM.ArticleDataset(X,y)
loader = DataLoader(datset, batch_size=16, shuffle=True)
generator = torch.Generator().manual_seed(67)

In [4]:
train_size = int(0.2 * len(datset))
valid_size = int(0.2 * len(datset))
test_size = len(datset) - train_size - valid_size

train_dataset, test_dataset, valid_dataset = random_split(datset, [train_size, valid_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)

In [5]:
lr = 1e-3
model = RM.ArticleLSTM()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task="multiclass", num_classes=3).to(device)

In [6]:
history = RM.train(
    device=device,
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    metric=metric,
    train_loader=train_loader,
    valid_loader=valid_loader,   # or val_loader, if you split 3 ways
    n_epochs=20,
)

Epoch 1/20,train loss: 0.6663, train metric: 70.83%, valid metric: 78.00%
Epoch 2/20,train loss: 0.5044, train metric: 80.03%, valid metric: 81.45%
Epoch 3/20,train loss: 0.4308, train metric: 83.29%, valid metric: 84.33%
Epoch 4/20,train loss: 0.3734, train metric: 85.96%, valid metric: 86.40%
Epoch 5/20,train loss: 0.3282, train metric: 88.18%, valid metric: 87.97%
Epoch 6/20,train loss: 0.2912, train metric: 89.66%, valid metric: 88.56%
Epoch 7/20,train loss: 0.2626, train metric: 90.79%, valid metric: 89.80%
Epoch 8/20,train loss: 0.2388, train metric: 91.54%, valid metric: 90.10%
Epoch 9/20,train loss: 0.2190, train metric: 92.32%, valid metric: 90.42%
Epoch 10/20,train loss: 0.2030, train metric: 92.96%, valid metric: 90.65%
Epoch 11/20,train loss: 0.1863, train metric: 93.59%, valid metric: 90.96%
Epoch 12/20,train loss: 0.1736, train metric: 93.91%, valid metric: 90.96%
Epoch 13/20,train loss: 0.1615, train metric: 94.36%, valid metric: 91.14%
Epoch 14/20,train loss: 0.1502, tr